# Prática 1 — O fator casa no Brasileirão

**Perguntas que vamos responder com dados:**
1. Qual a porcentagem de vitórias do mandante em cada temporada?
2. Ela caiu em 2020, o ano dos jogos sem torcida?

**Por que isso importa para o projeto:** o modelo de partidas precisa saber o quanto jogar em casa ajuda. Se essa vantagem muda com o tempo, talvez o modelo deva dar mais peso aos anos recentes.

**Como usar este notebook**
- Rode uma célula de cada vez com `Shift + Enter`.
- Onde aparecer `___`, é a sua vez: troque pelo código certo.
- Cada exercício tem uma **dica**. O **gabarito** está no fim, mas tente antes de olhar. Errar e ler a mensagem de erro faz parte do aprendizado.
- Rode o `01_coleta_dados.py` antes. O notebook pode ficar na pasta do projeto ou na subpasta `scripts`.

## Passo 0 — Carregar os dados

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# O notebook pode estar na pasta do projeto ou dentro de "scripts".
# Procuramos o arquivo nos dois lugares (".." significa "a pasta de cima").
candidatos = [Path("data/processed/partidas.csv"), Path("../data/processed/partidas.csv")]
CAMINHO = next((c for c in candidatos if c.exists()), None)
if CAMINHO is None:
    raise FileNotFoundError("Não achei o arquivo. Rode antes no terminal: python 01_coleta_dados.py")

# parse_dates transforma a coluna "data" em data de verdade (e não texto)
partidas = pd.read_csv(CAMINHO, parse_dates=["data"])
print("Linhas e colunas:", partidas.shape)

## Passo 1 — Conhecer a tabela

Antes de qualquer conta, sempre olhamos os dados. A tabela tem muitas colunas (odds, probabilidades...), mas hoje só precisamos de algumas. Para escolher colunas, passamos uma **lista de nomes** entre colchetes duplos: `tabela[["col1", "col2"]]`.

In [ ]:
colunas = ["temporada", "data", "mandante", "visitante", "gols_mandante", "gols_visitante", "resultado"]
partidas[colunas].head(10)

A coluna **`resultado`** foi criada pelo nosso script, sempre do ponto de vista do mandante:

| valor | significa |
|---|---|
| `M` | mandante venceu |
| `E` | empate |
| `V` | visitante venceu |

## Passo 2 — Contar os resultados (todas as temporadas juntas)

O método **`value_counts()`** conta quantas vezes cada valor aparece em uma coluna.

**Exercício 2a:** conte quantos jogos terminaram em M, E e V.

> Dica: o nome do método está no parágrafo acima. Não esqueça os parênteses.

In [ ]:
partidas["resultado"].___()

Números absolutos são difíceis de comparar. O mesmo método aceita o argumento `normalize`, que transforma a contagem em **proporção** (de 0 a 1).

**Exercício 2b:** mostre as proporções.

> Dica: `normalize` recebe um valor lógico do Python: `True` ou `False`.

In [ ]:
partidas["resultado"].value_counts(normalize=___)

✍️ **Para pensar:** o mandante vence mais ou menos que metade das vezes? E o visitante, quanto?

## Passo 3 — Separar por temporada com `groupby`

Imagine pegar todas as partidas e separar em pilhas, uma pilha por ano. Depois, fazer a mesma conta (`value_counts`) em cada pilha e juntar os resultados. É exatamente isso que o **`groupby`** faz. O padrão é:

```python
tabela.groupby("coluna_que_separa")["coluna_que_calcula"].conta()
```

**Exercício 3:** complete com a coluna que separa as partidas por ano.

> Dica: olhe a lista `colunas` do Passo 1.

In [ ]:
por_temporada = partidas.groupby("___")["resultado"].value_counts(normalize=True)
por_temporada.head(6)

O resultado funciona, mas está "empilhado": cada temporada aparece em três linhas. O método **`unstack()`** "desempilha", transformando os valores M, E e V em colunas. Fica uma linha por temporada, muito mais fácil de ler.

In [ ]:
tabela = por_temporada.unstack()

# Multiplicamos por 100 e arredondamos para ler como porcentagem
(tabela * 100).round(1)

## Passo 4 — Ver em um gráfico

O foco de hoje é pandas, então o código do gráfico já vem pronto. A faixa vermelha marca 2020: a temporada foi jogada entre agosto de 2020 e fevereiro de 2021, toda sem público.

⚠️ O ponto de 2026 ainda vai mudar: a temporada está em andamento e tem menos jogos que as outras.

In [ ]:
vitorias_mandante = tabela["M"] * 100
media = vitorias_mandante.mean()

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(vitorias_mandante.index, vitorias_mandante.values, marker="o", color="#1f5f8b")
ax.axhline(media, linestyle="--", color="gray", label=f"média do período ({media:.1f}%)")
if 2020 in vitorias_mandante.index:
    ax.axvspan(2019.5, 2020.5, color="red", alpha=0.12, label="2020, sem público")
ax.set_title("Vitórias do mandante por temporada (Série A)")
ax.set_ylabel("% dos jogos")
ax.set_xticks(vitorias_mandante.index)
ax.tick_params(axis="x", rotation=45)
ax.legend()
plt.tight_layout()
plt.show()

## Passo 5 — Interpretar (a parte mais importante)

Um gráfico pode enganar se olharmos só para o ponto que queremos ver. Vamos ordenar as temporadas da menor para a maior taxa de vitória do mandante.

**Exercício 5:** use o método que ordena valores.

> Dica: o nome começa com `sort_`. Ele ordena do menor para o maior por padrão.

In [ ]:
vitorias_mandante.___().round(1)

✍️ **Responda com suas palavras** (dê dois cliques nesta célula para escrever):

1. 2020 ficou abaixo da média?
2. Foi o menor valor de todos? Se não, qual foi?
3. Depois de 2020, a taxa voltou ao patamar de 2014–2016?
4. Dá para afirmar que "a falta de torcida derrubou o fator casa" só com esse gráfico? Por quê?

*Suas respostas:*

## Desafio extra — uma medida melhor: pontos por jogo

A porcentagem de vitórias ignora os empates. Uma medida mais completa é: **quantos pontos, em média, o mandante e o visitante ganham por jogo?** A diferença entre os dois é a "vantagem de casa em pontos".

O método **`map`** troca cada valor de uma coluna usando um dicionário. Por exemplo, `{"M": 3, "E": 1, "V": 0}` dá os pontos do mandante.

**Exercício extra:** complete o dicionário do visitante e a coluna do `groupby`.

> Dica: se o mandante venceu (`M`), quantos pontos o visitante ganha?

In [ ]:
partidas["pontos_mandante"] = partidas["resultado"].map({"M": 3, "E": 1, "V": 0})
partidas["pontos_visitante"] = partidas["resultado"].map({"M": ___, "E": ___, "V": ___})

pontos = partidas.groupby("___")[["pontos_mandante", "pontos_visitante"]].mean()
pontos["vantagem_casa"] = pontos["pontos_mandante"] - pontos["pontos_visitante"]
pontos.round(2)

✍️ **Para pensar:** a ordem dos anos muda quando usamos pontos em vez de % de vitórias? Qual temporada teve a menor vantagem de casa?

---

### Me conte o que você encontrou

Mande suas respostas do Passo 5 e do desafio. A partir delas, vamos decidir juntos como o modelo vai tratar o fator casa, por exemplo:
- um valor fixo, igual para todos os anos;
- um valor que muda com o tempo (dando mais peso aos anos recentes);
- um fator casa diferente para cada time (alguns estádios pesam mais que outros?).

---
# Gabarito

Só olhe depois de tentar! 🙂

**2a.** `partidas["resultado"].value_counts()`

**2b.** `partidas["resultado"].value_counts(normalize=True)`

**3.** `partidas.groupby("temporada")["resultado"].value_counts(normalize=True)`

**5.** `vitorias_mandante.sort_values().round(1)`

**Extra.**
```python
partidas["pontos_visitante"] = partidas["resultado"].map({"M": 0, "E": 1, "V": 3})
pontos = partidas.groupby("temporada")[["pontos_mandante", "pontos_visitante"]].mean()
```

**Erros comuns**
- `AttributeError: 'Series' object has no attribute 'value_count'` → faltou o "s" no fim: `value_counts`.
- `KeyError: 'Temporada'` → nomes de colunas diferenciam maiúsculas de minúsculas.
- Esquecer os parênteses: `value_counts` sem `()` mostra o *método*, e não o resultado.